# FreeSurfer on an HPC

**Author**: Steffen Bollmann

The University of Queensland<br>
<div style="line-height: 2;">
<a href="https://github.com/stebo85"><img src="https://img.shields.io/badge/-Steffen_Bollmann-181717?logo=github" alt="GitHub"></a> <a href="https://orcid.org/0000-0002-2909-0906"><img src="https://img.shields.io/badge/ORCID-0000--0002--2909--0906-green?logo=orcid" alt="ORCID"></a><br>
</div>

**Date**: 16/03/2026

**License:** 
<div style="margin-top: 10px;">
    <a href="https://opensource.org/licenses/MIT" target="_blank" style="color: #0066cc;">
        <i class="fas fa-balance-scale"></i> MIT License
    </a>
</div>

## Purpose

**FreeSurfer** is a widely used software suite for processing and analysing structural MRI data of the brain. Its flagship pipeline, `recon-all`, performs cortical reconstruction and volumetric segmentation - including skull stripping, white matter segmentation, cortical surface extraction, parcellation, and thickness estimation.

This tutorial demonstrates how to run FreeSurfer on an **HPC cluster** using Neurodesk's environment modules. This is the recommended approach for `recon-all`, which can take several hours to complete.

:::{admonition} Learning Objectives
:class: tip
By the end of this tutorial you will be able to:
- Load FreeSurfer via environment modules on an HPC
- Configure the required environment variables for containerised FreeSurfer
- Run the full `recon-all` cortical reconstruction pipeline
:::


## Citation and Resources

### Tools used in this workflow

__FreeSurfer__
: Fischl, B. (2012). FreeSurfer. *NeuroImage*, 62(2), 774–781. [https://doi.org/10.1016/j.neuroimage.2012.01.021](https://doi.org/10.1016/j.neuroimage.2012.01.021)

### Dataset

__TOMCAT Dataset__
: Available from the Open Science Framework, project [bt4ez](https://osf.io/bt4ez/).

### Educational resources

- [FreeSurfer Wiki](https://surfer.nmr.mgh.harvard.edu/fswiki)
- [FreeSurfer recon-all documentation](https://surfer.nmr.mgh.harvard.edu/fswiki/recon-all)
- [Neurodesk documentation](https://neurodesk.org)


## Prerequisites

:::{admonition} Before you begin
:class: warning
This tutorial assumes you have Neurodesk installed on your HPC. See the [Neurodesk HPC installation guide](https://neurodesk.org/getting-started/installations) for instructions.
:::

- [x] Neurodesk available on your HPC (environment modules configured)
- [ ] Familiarity with basic terminal commands and your HPC's job scheduler


## Download demo data

Download a T1-weighted image from the TOMCAT dataset:

```bash
pip install osfclient
osf -p bt4ez fetch osfstorage/TOMCAT_DIB/sub-01/ses-01_7T/anat/sub-01_ses-01_7T_T1w_defaced.nii.gz \
    sub-01_ses-01_7T_T1w_defaced.nii.gz
```


## Load FreeSurfer and configure environment

Load FreeSurfer using the environment module system:

```bash
ml freesurfer/7.3.2
```

Set up the required environment variables. Because FreeSurfer runs inside a Singularity/Apptainer container, you need to pass `SUBJECTS_DIR` into the container:

```bash
export SUBJECTS_DIR=$PWD/freesurfer-output
mkdir -p $SUBJECTS_DIR
export SINGULARITYENV_SUBJECTS_DIR=$SUBJECTS_DIR
export APPTAINERENV_SUBJECTS_DIR=$SUBJECTS_DIR
```

:::{note}
Both `SINGULARITYENV_` and `APPTAINERENV_` prefixes are set for compatibility with older (Singularity) and newer (Apptainer) container runtimes.
:::


## Run recon-all

Launch the full cortical reconstruction pipeline:

```bash
recon-all -subject test-subject \
    -i sub-01_ses-01_7T_T1w_defaced.nii.gz \
    -all
```

:::{note}
`recon-all -all` runs the complete FreeSurfer pipeline. This typically takes **6–12 hours** depending on your hardware. On an HPC, consider submitting this as a batch job rather than running it interactively.
:::


### Using FreeSurfer 8.0.0 or later

If you are using FreeSurfer version 8.0.0 or newer, you need to enable the deep learning modules before running `recon-all`:

```bash
export FS_ALLOW_DEEP=1
```

If running on a GPU, also ensure the following is set:

```bash
export neurodesk_singularity_opts='--nv'
```


## Inspect the results

Once `recon-all` completes, the outputs will be in `$SUBJECTS_DIR/test-subject/`. Verify:

```bash
ls $SUBJECTS_DIR/test-subject/
```

Key output directories include:

`surf/`
: Cortical surfaces (pial, white, inflated)

`label/`
: Parcellations (Desikan-Killiany, Destrieux)

`stats/`
: Volumetric and surface statistics

`mri/`
: Processed volumes (brain mask, segmentations, bias-corrected T1)

You can use **FreeView** (included with FreeSurfer) to visualise the results:

```bash
freeview -v $SUBJECTS_DIR/test-subject/mri/T1.mgz \
    -f $SUBJECTS_DIR/test-subject/surf/lh.pial:edgecolor=red \
    $SUBJECTS_DIR/test-subject/surf/lh.white:edgecolor=blue
```


## Summary

In this tutorial you:

1. Downloaded a demo T1-weighted MRI dataset from the TOMCAT project
2. Loaded FreeSurfer via environment modules on an HPC
3. Configured the container environment variables
4. Ran the full `recon-all` cortical reconstruction pipeline

:::{seealso}
- [FreeSurfer in Neurodesktop](freesurfer.ipynb) - for running FreeSurfer via the Neurodesk application with a GUI
:::
